# ⚙️ Hyperparameter Search for Fusion Classifiers — K=4

This notebook is part of the article **“DCMC-Fusion: Data-Centric and Model-Centric Ensemble Learning for Audio Deepfake Detection”**, by **Dora Ballesteros and Carlos Guerra**.

Its purpose is to identify **one optimal hyperparameter configuration for each fusion classifier at K=4**. The selected hyperparameters remain fixed across all five architecture combinations and are subsequently used in the corresponding DCMC-Fusion experiment.

### 📥 Input File

This notebook requires **one input file**, generated prior to and independently of the DCMC-Fusion method:

1. **`inference_probabilities.csv`:** contains the inference probabilities generated by 20 previously trained models, corresponding to five architectures (ResNet50, DenseNet121, InceptionV3, ResNet34, and VGG16) and four spectral representations (Mel, LOG, DWT, and CQT). Each row represents an audio sample, with probability values ranging from 0 to 100. The `label` column provides the ground-truth class (0 = natural, 1 = synthetic).

### 🔬 Hyperparameter Search Procedure

- **Ensemble configuration:** for **K=4**, five datasets are constructed from all possible combinations of four out of the five architectures. Each dataset contains **16 probability features**, corresponding to the four spectral representations (Mel, LOG, DWT, and CQT) of each selected architecture.

- **Data partition:** a stratified **70/30 training–test split** is performed using `random_state=42`. Hyperparameter selection uses only the training subset, while the test subset is reserved for subsequent experiments.

- **Fusion classifiers:** Logistic Regression, Random Forest, XGBoost, and SVM.

- **Search procedure:** each hyperparameter candidate is evaluated using **five-fold cross-validation**, with **macro F1** as the optimization metric. `StandardScaler` is included within each pipeline and fitted exclusively on the training data of each fold.

- **Hyperparameter selection:** for each classifier, the optimal configuration is selected by maximizing the **mean cross-validation macro F1 across all five architecture combinations**. Thus, a single hyperparameter configuration is selected per classifier rather than independently optimizing each architecture combination.

### 📤 Output Files

This notebook generates **three output files**:

1. **`best_hyperparameters_k4.json`:** stores the optimal hyperparameter configuration selected for each of the four fusion classifiers. These configurations are subsequently loaded by `DCMC_Fusion_K4.ipynb`, without repeating the hyperparameter search.

2. **`experiment_split.json`:** stores the training and test indices, the audio sample order, the ground-truth labels, and additional information required to reproduce and verify the data partition. The same partition is used across all ensemble sizes (K=1 to K=5).

3. **`hyperparameter_search_summary_k4.csv`:** summarizes the hyperparameter search results, including the selected configuration and its corresponding mean cross-validation macro F1 score for each fusion classifier.

### 🔗 Subsequent Experiment

The selected hyperparameters are used in **`DCMC_Fusion_K4.ipynb`**, where each fusion classifier and its associated scaler are independently fitted for each of the five architecture combinations using the training partition and evaluated on the held-out test partition.

This separation ensures that **hyperparameter optimization and final performance evaluation are conducted independently**, avoiding the use of test data during model selection.

In [ ]:
# Install dependencies
%pip install numpy pandas scikit-learn xgboost --quiet

In [ ]:
# Data handling and output
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Preprocessing and hyperparameter search
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

# Fusion classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.30
CV_FOLDS = 5
SCORING = "f1_macro"

np.random.seed(RANDOM_STATE)



In [ ]:
#-------------------------------------------------------
# Choose ensemble size here (1, 2, 3, 4, or 5)
#--------------------------------------------------------

K = 4
if not isinstance(K, int) or K not in range(1, 6):
    raise ValueError("K must be an integer between 1 and 5.")
print(f"Selected K = {K}")


Selected K = 4


In [ ]:
# Load inference probabilities
CSV_PATH = Path("inference_probabilities.csv")
df = pd.read_csv(CSV_PATH)

# Five architectures, each with four spectral representations
ARCHITECTURES = {
    "resnet50": [
        "resnet50_mel", "resnet50_log",
        "resnet50_dwt", "resnet50_cqt"
    ],
    "densenet121": [
        "densenet121_mel", "densenet121_log",
        "densenet121_dwt", "densenet121_cqt"
    ],
    "inception_v3": [
        "inception_v3_mel", "inception_v3_log",
        "inception_v3_dwt", "inception_v3_cqt"
    ],
    "resnet34": [
        "resnet34_mel", "resnet34_log",
        "resnet34_dwt", "resnet34_cqt"
    ],
    "vgg16": [
        "vgg16_mel", "vgg16_log",
        "vgg16_dwt", "vgg16_cqt"
    ]
}

FEATURE_COLUMNS = [
    column
    for columns in ARCHITECTURES.values()
    for column in columns
]

# Validate input columns
required_columns = ["audio", "label"] + FEATURE_COLUMNS
missing_columns = sorted(set(required_columns) - set(df.columns))

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Input features and ground truth
X = df[FEATURE_COLUMNS].to_numpy(dtype=float)
y = df["label"].to_numpy()

if not np.isfinite(X).all() or ((X < 0) | (X > 100)).any():
    raise ValueError("Probabilities must be finite and in [0, 100].")

if set(np.unique(y)) != {0, 1}:
    raise ValueError("Labels must contain 0 (natural) and 1 (synthetic).")

# Split row indices to preserve sample identities
train_indices, test_indices = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = X[train_indices]
y_train = y[train_indices]

# The test subset is reserved and is not used in the search
print(f"Input features: {X.shape[1]}")
print(f"Training samples: {len(train_indices)}")
print(f"Reserved test samples: {len(test_indices)}")

display(
    pd.Series(y_train)
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .to_frame("training_samples")
)

Input features: 20
Training samples: 1738
Reserved test samples: 746


,training_samples
label,
0,869
1,869


In [ ]:
# Classifiers and hyperparameter grids
ML_MODELS = {
    "Logistic Regression": (
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ),
        {
            "model__C": [0.01, 0.1, 1, 10],
            "model__penalty": ["l1", "l2"],
            "model__solver": ["liblinear", "saga"]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            "model__n_estimators": [100, 300, 700],
            "model__max_depth": [3, 5, 10, None],
            "model__min_samples_split": [3, 5, 8]
        }
    ),

    "XGBoost": (
        XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            verbosity=0,
            n_jobs=1
        ),
        {
            "model__n_estimators": [100, 300, 700],
            "model__learning_rate": [0.01, 0.1, 1],
            "model__max_depth": [3, 5, 8]
        }
    ),

    "SVM": (
        SVC(
            probability=True,
            random_state=RANDOM_STATE
        ),
        {
            "model__C": [0.1, 1, 5, 10],
            "model__gamma": ["scale", "auto"],
            "model__kernel": ["rbf"]
        }
    )
}

In [ ]:
# Select parameters by mean CV F1-macro across every architecture subset of size K
from itertools import combinations
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_val_score

architecture_combinations = list(combinations(ARCHITECTURES, K))
# Fix folds so every candidate and subset is evaluated on identical partitions
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=False)

best_hyperparameters = {}
search_summary = []

for name, (classifier, parameter_grid) in ML_MODELS.items():
    print(f"\nSearching hyperparameters for K={K}: {name}", flush=True)
    candidates = list(ParameterGrid(parameter_grid))
    candidate_means = []

    for candidate_number, candidate in enumerate(candidates, start=1):
        subset_scores = []
        for architecture_subset in architecture_combinations:
            subset_columns = [
                column
                for architecture in architecture_subset
                for column in ARCHITECTURES[architecture]
            ]
            X_subset = df[subset_columns].to_numpy(dtype=float)[train_indices]

            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", clone(classifier))
            ])
            pipeline.set_params(**candidate)
            scores = cross_val_score(
                pipeline, X_subset, y_train,
                scoring=SCORING, cv=cv, n_jobs=-1,
                error_score="raise"
            )
            subset_scores.append(float(np.mean(scores)))

        candidate_means.append(float(np.mean(subset_scores)))

    # np.argmax chooses the first candidate in case of ties
    best_index = int(np.argmax(candidate_means))
    best_candidate = candidates[best_index]
    parameters = {
        key.removeprefix("model__"): value
        for key, value in best_candidate.items()
    }
    best_hyperparameters[name] = parameters
    search_summary.append({
        "K": K,
        "classifier": name,
        "architecture_combinations": len(architecture_combinations),
        "best_cv_f1_macro": candidate_means[best_index],
        "best_parameters": json.dumps(parameters, sort_keys=True)
    })
    print(f"Best parameters: {parameters}")
    print(f"Mean CV macro F1: {candidate_means[best_index]:.4f}")

search_summary_df = pd.DataFrame(search_summary)
display(search_summary_df)



Searching hyperparameters for K=4: Logistic Regression
Best parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Mean CV macro F1: 0.7142

Searching hyperparameters for K=4: Random Forest
Best parameters: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 700}
Mean CV macro F1: 0.7190

Searching hyperparameters for K=4: XGBoost
Best parameters: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300}
Mean CV macro F1: 0.7193

Searching hyperparameters for K=4: SVM
Best parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Mean CV macro F1: 0.7044


,K,classifier,architecture_combinations,best_cv_f1_macro,best_parameters
0,4,Logistic Regression,5,0.714198,"{""C"": 0.1, ""penalty"": ""l2"", ""solver"": ""libline..."
1,4,Random Forest,5,0.718967,"{""max_depth"": 5, ""min_samples_split"": 5, ""n_es..."
2,4,XGBoost,5,0.719328,"{""learning_rate"": 0.01, ""max_depth"": 3, ""n_est..."
3,4,SVM,5,0.704361,"{""C"": 0.1, ""gamma"": ""scale"", ""kernel"": ""rbf""}"


In [ ]:
# Save using exactly the filenames and JSON structures expected by all K notebooks
with open("best_hyperparameters_k4.json", "w", encoding="utf-8") as file:
    json.dump(best_hyperparameters, file, indent=4)

search_summary_df.to_csv("hyperparameter_search_summary.csv", index=False)

# IMPORTANT: full 20-column list is required by the unchanged DCMC_Fusion_K*.ipynb notebooks.
# 'search_K' records the actual subset size used for hyperparameter optimization.
split_configuration = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "scoring": SCORING,
    "search_K": K,
    "feature_columns": FEATURE_COLUMNS,
    "train_indices": train_indices.tolist(),
    "test_indices": test_indices.tolist(),
    "audio_order": df["audio"].astype(str).tolist(),
    "label_order": y.tolist()
}

with open("experiment_split.json", "w", encoding="utf-8") as file:
    json.dump(split_configuration, file, indent=4)

print(f"Search completed for K={K}.")
print("Saved: best_hyperparameters_k4.json")
print("Saved: hyperparameter_search_summary.csv")
print("Saved: experiment_split.json")


Search completed for K=4.
Saved: best_hyperparameters_k4.json
Saved: hyperparameter_search_summary.csv
Saved: experiment_split.json
